# TP 3 — Préparer un jeu de données avec Spark

**M1 · Deep Learning & Big Data · Séance 3 · Environnement : Google Colab**

On prépare un vrai jeu de données de **50 000 critiques de films** (IMDb), étiquetées positives ou négatives. Ce jeu propre servira aux TP 4 et 6 pour entraîner et tester des modèles de texte.

Pour rendre l'exercice réaliste, on commence par **simuler une collecte imparfaite** : valeurs manquantes, doublons, étiquettes écrites de plusieurs façons, balises HTML… C'est le quotidien d'un data engineer.

| Étape | Ce qu'on fait |
|---|---|
| 1 | Charger les données brutes |
| 2 | Explorer : repérer les défauts |
| 3 | Nettoyer : manquants, étiquettes, doublons |
| 4 | Nettoyer le texte |
| 5 | Créer des colonnes utiles et filtrer |
| 6 | Découper en entraînement / test et enregistrer en Parquet |
| 7 | Appliquer la démarche à votre projet fil rouge |

Complétez les `____` des cellules `# TODO` et répondez aux questions ❓.

---
## Partie 0 — Installation

In [ ]:
import shutil
!pip install -q pyspark datasets
if shutil.which("java") is None:
    !apt-get install -y -qq openjdk-17-jre-headless > /dev/null

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("TP3").config("spark.driver.memory", "4g").getOrCreate()
print("Spark", spark.version, "prêt ✅")

---
## Partie 1 — Charger les données brutes

Les critiques IMDb sont téléchargées depuis le Hub Hugging Face. Puis la cellule suivante **abîme volontairement** les données pour simuler une collecte réelle. Lisez-la pour savoir quels défauts chercher… ou faites semblant de ne pas l'avoir lue pour jouer le jeu de l'exploration !

In [ ]:
from datasets import load_dataset
import pandas as pd

imdb = load_dataset("stanfordnlp/imdb")
df = pd.concat([imdb["train"].to_pandas(), imdb["test"].to_pandas()], ignore_index=True)
df = df.rename(columns={"text": "texte", "label": "label"})
print(len(df), "critiques chargées")
df.head(3)

In [ ]:
# --- Simulation d'une collecte imparfaite (ne pas modifier) ---
import numpy as np
rng = np.random.default_rng(0)

brut = df.copy()
# 1. Étiquettes saisies de plusieurs façons
variantes = {0: ["neg", "negative", "0", " Négatif", "NEG"], 1: ["pos", "positive", "1", "Positif ", "POS"]}
brut["label"] = [rng.choice(variantes[l]) for l in brut["label"]]
# 2. Quelques étiquettes illisibles
brut.loc[rng.choice(len(brut), 150, replace=False), "label"] = "???"
# 3. Textes manquants ou vides
brut.loc[rng.choice(len(brut), 400, replace=False), "texte"] = None
brut.loc[rng.choice(len(brut), 200, replace=False), "texte"] = "   "
# 4. Doublons (une même critique collectée deux fois)
brut = pd.concat([brut, brut.sample(1500, random_state=1)], ignore_index=True)
# 5. Une colonne source et un identifiant
brut["source"] = rng.choice(["site_web", "appli_mobile", "partenaire"], len(brut))
brut.insert(0, "id", np.arange(len(brut)))

brut.to_parquet("imdb_brut.parquet", index=False)
print(len(brut), "lignes brutes écrites dans imdb_brut.parquet")

In [ ]:
avis = spark.read.parquet("imdb_brut.parquet")
avis.printSchema()
avis.show(5, truncate=60)

---
## Partie 2 — Explorer avant de nettoyer

Règle d'or : **on ne nettoie pas à l'aveugle**. On mesure d'abord les défauts.

### 🛠️ Exercice 2.1 — Valeurs des étiquettes

Affichez toutes les valeurs distinctes de la colonne `label` avec leur nombre d'occurrences.

In [ ]:
avis.groupBy(____).count().orderBy(F.desc("count")).show()   # TODO

### 🛠️ Exercice 2.2 — Valeurs manquantes

Comptez, pour chaque colonne, le nombre de valeurs nulles.

> 💡 `F.col(c).isNull()` vaut vrai pour une valeur manquante ; `F.when(condition, 1)` vaut 1 quand la condition est vraie ; `F.count()` compte les valeurs non nulles.

In [ ]:
avis.select([
    F.count(F.when(F.col(c).____(), 1)).alias(c)   # TODO : tester si la valeur est nulle
    for c in avis.columns
]).show()

In [ ]:
# Doublons : combien de textes apparaissent plusieurs fois ?
(avis.groupBy("texte").count()
     .filter(F.col("count") > 1)
     .agg(F.count("*").alias("textes_en_double"), F.sum("count").alias("lignes_concernees"))
     .show())

In [ ]:
# Des balises HTML dans le texte ?
avis.filter(F.col("texte").contains("<br")).select("texte").show(3, truncate=120)
print("Critiques contenant <br :", avis.filter(F.col("texte").contains("<br")).count())

### ❓ Question 2.3

Listez les défauts que vous avez repérés et, pour chacun, la correction que vous proposez.

*Votre réponse :*

---
## Partie 3 — Nettoyer : manquants, étiquettes, doublons

### 🛠️ Exercice 3.1 — Supprimer les textes vides

Supprimez les lignes dont le texte est nul **ou** ne contient que des espaces.

> 💡 `dropna(subset=[...])` supprime les lignes où ces colonnes sont nulles ; `F.trim()` retire les espaces au début et à la fin.

In [ ]:
etape1 = (avis
          .dropna(subset=[____])                    # TODO
          .filter(F.trim(F.col("texte")) != ____))  # TODO
print("Après suppression des textes vides :", etape1.count())

### 🛠️ Exercice 3.2 — Normaliser les étiquettes

Transformez toutes les variantes en `1` (positif) ou `0` (négatif). Les valeurs inconnues deviennent nulles et sont supprimées.

> 💡 On met d'abord tout en minuscules sans espaces, puis `F.when(...).when(...).otherwise(None)` fonctionne comme un `if / elif / else`.

In [ ]:
label_net = F.lower(F.trim(F.col("label")))

etape2 = (etape1
          .withColumn("label",
                      F.when(label_net.isin("pos", "positive", "positif", "1"), 1)
                       .when(label_net.isin(____), 0)                 # TODO : les variantes négatives
                       .otherwise(None))
          .dropna(subset=["label"]))
etape2.groupBy("label").count().show()

### 🛠️ Exercice 3.3 — Supprimer les doublons

Gardez une seule ligne par texte.

In [ ]:
etape3 = etape2.____(["texte"])   # TODO
print("Après dédoublonnage :", etape3.count())

---
## Partie 4 — Nettoyer le texte

Les **expressions régulières** (regex) décrivent des motifs de texte. Deux suffisent ici :
- `<[^>]+>` : une balise HTML (un `<`, puis tout sauf `>`, puis `>`) ;
- `\s+` : une suite d'espaces, tabulations ou retours à la ligne.

### 🛠️ Exercice 4.1

Créez la colonne `texte_propre` : balises HTML remplacées par un espace, tout en minuscules, espaces multiples réduits à un seul, espaces de début et de fin retirés.

In [ ]:
etape4 = etape3.withColumn(
    "texte_propre",
    F.trim(
        F.regexp_replace(
            F.____(F.regexp_replace("texte", r"<[^>]+>", " ")),   # TODO : mettre en minuscules
            r"\s+", ____)                                       # TODO : remplacer par un seul espace
    ),
)
etape4.select("texte", "texte_propre").show(3, truncate=70)

In [ ]:
# Vérification : il ne doit plus rester de balise
print("Critiques avec encore une balise :", etape4.filter(F.col("texte_propre").rlike("<[^>]+>")).count())

---
## Partie 5 — Créer des colonnes utiles et filtrer

### 🛠️ Exercice 5.1

Ajoutez la colonne `nb_mots` (nombre de mots de `texte_propre`), puis retirez les critiques de **moins de 10 mots**, trop courtes pour être utiles.

> 💡 `F.split(col, " ")` découpe en liste de mots ; `F.size(liste)` donne sa longueur.

In [ ]:
etape5 = (etape4
          .withColumn("nb_mots", F.size(F.split(____, " ")))   # TODO
          .filter(F.col("nb_mots") >= ____))                   # TODO
etape5.select("nb_mots").summary("min", "25%", "50%", "75%", "max").show()

In [ ]:
# Équilibre des classes et longueur moyenne par classe
total = etape5.count()
(etape5.groupBy("label")
       .agg(F.count("*").alias("nb"), F.round(F.avg("nb_mots"), 1).alias("mots_moyens"))
       .withColumn("part", F.round(F.col("nb") / total, 3))
       .show())

### ❓ Question 5.2

Les deux classes sont-elles **équilibrées** ? Pourquoi est-ce important pour entraîner un modèle de classification ?

*Votre réponse :*

---
## Partie 6 — Découper et enregistrer

On découpe en **80 % entraînement / 20 % test**, de façon reproductible (`seed`), puis on écrit en **Parquet partitionné** par `split` : un sous-dossier par partie.

In [ ]:
final = etape5.select("id", "texte_propre", "label", "nb_mots", "source")

train, test = final.randomSplit([____, ____], seed=42)   # TODO : proportions

jeu = (train.withColumn("split", F.lit("train"))
       .unionByName(test.withColumn("split", F.lit("test"))))

jeu.write.mode("overwrite").partitionBy("split").parquet("imdb_propre")
print("Entraînement :", train.count(), "· Test :", test.count())

In [ ]:
# Ce que Spark a écrit sur le disque : un dossier par split, plusieurs fichiers par dossier
!find imdb_propre -maxdepth 2 | head -20

### ❓ Question 6.1

Pourquoi Spark écrit-il **plusieurs fichiers** `part-xxxx.parquet` au lieu d'un seul ?

*Votre réponse :*

### 💾 Sauvegarder sur Google Drive (recommandé)

Les fichiers de Colab disparaissent à la fermeture de la session. Copiez `imdb_propre` sur votre Drive pour le retrouver aux TP 4 et 6.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/DeepLearning"
!cp -r imdb_propre "/content/drive/MyDrive/DeepLearning/"
print("Copié dans MyDrive/DeepLearning/imdb_propre ✅")

---
## Partie 7 — À vous : votre projet fil rouge

Appliquez la même démarche à **votre** jeu de données. Complétez le tableau puis écrivez le pipeline dans les cellules suivantes.

| Étape | Ce que vous avez constaté | Ce que vous faites |
|---|---|---|
| Valeurs manquantes | | |
| Doublons | | |
| Étiquettes / catégories | | |
| Format du texte ou des images | | |
| Équilibre des classes | | |
| Découpage train / test | | |

In [ ]:
# Votre pipeline de préparation
# projet = spark.read.parquet(...) ou spark.read.csv(..., header=True, inferSchema=True)

---
## 🎯 Pour vérifier que vous avez compris

1. Pourquoi explore-t-on les données avant de les nettoyer ?
2. À quoi sert le paramètre `seed` de `randomSplit` ?
3. Quel avantage a le format Parquet par rapport au CSV pour la suite du projet ?

*Vos réponses :*

In [ ]:
spark.stop()